# 2 - Python + R

Persistent session, typed data transfer, plots, and R's warnings and errors
arriving as Python objects.

In [ ]:
%load_ext econenv

In [ ]:
import numpy as np
import pandas as pd
import econenv

rng = np.random.default_rng(20260904)
n = 120
df = pd.DataFrame({'x1': rng.normal(size=n), 'x2': rng.normal(size=n)})
df['y'] = 2.0 + 0.5 * df.x1 - 0.3 * df.x2 + rng.normal(scale=0.4, size=n)
df.head()

## Push, model, pull back

`-o` brings an R object home as a DataFrame - with the term names as its index.

In [ ]:
%%R -i df -o coef_table
fit <- lm(y ~ x1 + x2, data = df)
coef_table <- as.data.frame(coef(summary(fit)))

In [ ]:
coef_table

## The session persists between cells

In [ ]:
%%R
confint(fit)

## Plots come back inline

In [ ]:
%%R
plot(fitted(fit), resid(fit), pch = 20,
     xlab = 'Fitted', ylab = 'Residual', main = 'Residuals vs fitted')
abline(h = 0, lty = 2)

## Warnings stay separate from output

In [ ]:
result = econenv.engine('r').execute("warning('mind out'); cat('body')")
print('stdout  :', result.stdout)
print('warnings:', result.warnings)

## Errors are Python exceptions, and the session survives

In [ ]:
%%R
survivor <- 42

In [ ]:
try:
    econenv.engine('r').execute("stop('deliberate failure')")
except Exception as exc:
    print(type(exc).__name__, exc)

In [ ]:
%%R
cat('still here:', survivor)